# MyCON — تنها نوت‌بوک اجرایی برای پایان‌نامه

**روش منتخب:** COLMAP + gsplat رسمی 1.5.3، مطابق تجربه بازسازی از ویدئوی واقعی موبایل در SpatialScene3D. خروجی هندسی COLMAP برای اعتبارسنجی مستقل BIM نگهداری می‌شود؛ Gaussian PLY به تنهایی اسکن متریک نیست.

۱. Runtime → GPU (مثلاً T4) را انتخاب کن.
۲. سلول اصلی را با `BACKEND = "colmap_gsplat"` و `MODE = "preflight"` اجرا کن؛ این مرحله ZIP اصلی، استخراج فریم‌ها، ثبت دوربین با COLMAP و آزمون CUDA را بررسی می‌کند؛ آموزش انجام نمی‌دهد.
۳. سلول وضعیت را اجرا کن. فقط با `PREFLIGHT_COLMAP_AND_GSPLAT_CUDA_PASSED_NO_TRAINING`، حالت `MODE = "all"` را برای اجازه آموزش انتخاب کن.
۴. LongSplat در کد محفوظ است و فقط برای آزمایش مقایسه‌ای می‌توان `BACKEND = "longsplat"` انتخاب کرد؛ مسیر پیشنهادی نیست.
داده خام Recorder، pose.csv، IMU و checkpointهای قدیمی تغییر یا حذف نمی‌شوند. بدون نقاط کنترل مستقل، دقت BIM تأیید نمی‌شود.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

# فقط در اجرای اول «preflight». برای شروع واقعی آموزش «all» بگذارید.
BACKEND = "colmap_gsplat"  # Selected: robust phone-video COLMAP + official gsplat v1.5.3
MODE = "preflight"  # preflight | all (train is LongSplat research only)
CAPTURE_ZIP = ""    # فقط اگر جستجوی خودکار ZIP اصلی را پیدا نکرد
ATTEMPT = "baseline_20261105"
QUALITY = "quality"  # برای T4: 8 FPS، عرض 512، 20000 iterations

drive.mount('/content/drive')  # اتصال مجدد به Drive؛ خروج از حساب انجام نمی‌شود
repo = Path('/content/MyCON_FINAL_ACCEPTANCE')
branch = 'feature/mycon-single-final-acceptance-20261105'
url = 'https://github.com/sayyedalimrj/MyCon.git'
if repo.exists():
    current = subprocess.check_output(['git','-C',str(repo),'remote','get-url','origin'],text=True).strip()
    assert current in (url, 'git@github.com:sayyedalimrj/MyCon.git'), 'Existing directory is not the expected MyCON repository'
    subprocess.run(['git','-C',str(repo),'fetch','origin',branch],check=True)
    subprocess.run(['git','-C',str(repo),'checkout',branch],check=True)
    subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',branch],check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',branch,url,str(repo)],check=True)
if BACKEND == "colmap_gsplat":
    import shutil
    if shutil.which("colmap") is None:
        subprocess.run(["apt-get","update","-qq"],check=True)
        subprocess.run(["apt-get","install","-y","colmap"],check=True)
    script = repo/'repository/scripts/mycon_colmap_gsplat.py'
    cmd = ['python',str(script),'--mode',MODE,'--attempt',ATTEMPT]
    if MODE == 'all': cmd.append('--approve-training')
elif BACKEND == "longsplat":
    script = repo/'repository/scripts/mycon_thesis_one_run.py'
    cmd = ['python',str(script),'--mode',MODE,'--attempt',ATTEMPT,
           '--quality',QUALITY,'--accept-research-license']
else:
    raise ValueError("BACKEND must be 'colmap_gsplat' or 'longsplat'")
if CAPTURE_ZIP.strip(): cmd += ['--capture',CAPTURE_ZIP.strip()]
print('BACKEND:',BACKEND,'| MODE:',MODE,'| TRAINING:',MODE in ('all','train'))
subprocess.run(cmd,check=True)


## گزارش وضعیت (بدون شروع دوباره آموزش)
این سلول فقط خلاصه وضعیت همان Attempt را می‌خواند؛ هیچ GPU training راه‌اندازی نمی‌کند.

In [ ]:
from pathlib import Path
import subprocess
BASE = Path('/content/MyCON_FINAL_ACCEPTANCE/repository/scripts')
if globals().get("BACKEND", "colmap_gsplat") == "colmap_gsplat":
    command = ['python',str(BASE/'mycon_colmap_gsplat.py'),
               '--mode','status','--attempt',globals().get("ATTEMPT","baseline_20261105")]
else:
    command = ['python',str(BASE/'mycon_thesis_one_run.py'),
               '--mode','status','--attempt',globals().get("ATTEMPT","stabilization_20261105")]
subprocess.run(command,check=False)
